***Q1: Implementing an RNN for Text Generation***

Task: Recurrent Neural Networks (RNNs) can generate sequences of text. You will train an LSTM-based RNN to predict the next character in a given text dataset.

- Load a text dataset (e.g., "Shakespeare Sonnets", "The Little Prince").
- Convert text into a sequence of characters (one-hot encoding or embeddings).
- Define an RNN model using LSTM layers to predict the next character.
- Train the model and generate new text by sampling characters one at a time.
- Explain the role of temperature scaling in text generation and its effect on randomness.

In [45]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

In [46]:
# load dataset

path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

text = open(path, "rb").read().decode("utf-8")

print("Number of characters:", len(text))
print(text[:500])

Number of characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
# covert text into charcaters 

vocab = sorted(set(text))
char_to_index = {
    char: indx 
    for indx, char in enumerate(vocab)
}

index_to_char = np.array(vocab)

In [48]:
char_to_index

{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}

In [49]:
index_to_char

array(['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?',
       'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
       'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
       'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
       'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'],
      dtype='<U1')

In [ ]:
# convert the complete text

text_as_int = np.array(
    [char_to_index[c] for c in text],
    dtype=np.int32
)


In [52]:
text_as_int[:50]

array([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43,
       44, 53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39,
       52, 63,  1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56],
      dtype=int32)

In [53]:
# create traing sequence

sequence_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(
    sequence_length + 1,
    drop_remainder=True
)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]

    return input_text, target_text

dataset = sequences.map(split_input_target)

In [54]:
# sequence

for input_example, target_example in dataset.take(1):

    print("INPUT:")
    print("".join(index_to_char[input_example.numpy()]))

    print("\nTARGET:")
    print("".join(index_to_char[target_example.numpy()]))

INPUT:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

TARGET:
irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You 


In [ ]:
# batch and shuffle the dataset 

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

In [56]:
# lstm model

vocab_size = len(vocab)

embedding_dim = 256
lstm_units = 512

model = Sequential([
    
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    ),

    LSTM(
        lstm_units,
        return_sequences=True
    ),

    Dense(vocab_size)
])

model.build(input_shape=(None, sequence_length))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 256)       │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100, 512)       │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100, 65)        │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,624,897 (6.20 MB)

 Trainable params: 1,624,897 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#compile the model

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

model.compile(
    optimizer="adam",
    loss=loss_fn
)

In [59]:
# train the model

EPOCHS = 20

history = model.fit(
    dataset,
    epochs=EPOCHS
)

Epoch 1/20


/Users/supriiyaaa/UCMO/Fall 2026/Neural Network/Home_Assignment_2/env/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 ━━━━━━━━━━━━━━━━━━━━ 40s 226ms/step - loss: 2.5038
Epoch 2/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 42s 244ms/step - loss: 1.8949
Epoch 3/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 44s 256ms/step - loss: 1.6889
Epoch 4/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 47s 269ms/step - loss: 1.5735
Epoch 5/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 53s 307ms/step - loss: 1.5002
Epoch 6/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 55s 315ms/step - loss: 1.4466
Epoch 7/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 328ms/step - loss: 1.4069
Epoch 8/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 58s 332ms/step - loss: 1.3759
Epoch 9/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 327ms/step - loss: 1.3495
Epoch 10/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 58s 333ms/step - loss: 1.3274
Epoch 11/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 58s 333ms/step - loss: 1.3067
Epoch 12/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 329ms/step - loss: 1.2881
Epoch 13/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 332ms/step - loss: 1.2716
Epoch 14/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 60s 348ms/step - loss: 1.2553
Epoch 15/20
172/172 ━━━━━━

Temperature  controls the randomness of predictions by scaling the logits before applying the softmax function:
- Low Temperature (T -> 0): Sharpens the probability distribution. The model becomes very confident and picks the most likely character, resulting in repetitive, predictable text.
- High Temperature ($T -> 1+): Flattens the distribution, giving lower-probability characters a fairer chance. This increases creativity and diversity, but can lead to gibberish or spelling mistakes.
- Neutral Temperature (T = 1.0): Uses the raw model probabilities as it is.

In [ ]:
# generate text function
def generate_text(
    model,
    start_string,
    num_generate=500,
    temperature=1.0
):

    input_ids = [
        char_to_index[c]
        for c in start_string
        if c in char_to_index
    ]

    input_ids = tf.expand_dims(input_ids, 0)

    generated_text = list(start_string)

    for _ in range(num_generate):

        predictions = model(input_ids, training=False)

        # only predict for last character
        predictions = predictions[:, -1, :]

        # temperature scaling
        predictions = predictions / temperature

        # sample a character
        predicted_id = tf.random.categorical(
            predictions,
            num_samples=1
        )[0, 0].numpy()

        predicted_char = index_to_char[predicted_id]

        generated_text.append(predicted_char)

        # add predicted character to input
        predicted_id_tensor = tf.expand_dims(
            [predicted_id],
            0
        )

        input_ids = tf.concat(
            [input_ids, predicted_id_tensor],
            axis=1
        )

        # keep sequence length manageable
        input_ids = input_ids[:, -sequence_length:]

    return "".join(generated_text)

In [ ]:
# generate text
generated = generate_text(
    model,
    start_string="ROMEO:",
    num_generate=1000,
    temperature=1.0
)

print(generated)

ROMEO:
She is too grame, he's a subtle terthing else.

SICINIUS:
How now, no; and I gall of pullay!

LUCIO:
Juintle gen.

BRUTUS:
He has ade one that we should hand to take offer
As these thou hast might he slew howering and
these soldiers draws.'

CAMILLO:
He very slight,
Love, where he departard; put you, old walls and
Fortune's neck.

QUEEN MARGARET:
Give me the sea, them not wash my country,
For the people's strange outward swords:
That Clarence's friends, had any lie.
Grandam of last, better than rust; so blushfullights,
Or I am nothing stay: besides, 'tis his
time, unpitied. Now, changing Romeo live? Balths,
Hadst thou hear the tempest of against peace.

Clown:
Not dispatch'd honour!

BRAKENBURY:
Vinceling Richard, with her on thy sub, nurse.

GLOUCESTER:
The Duke of York, thou wert blood with whatefuls;
Forced that sleep burstorsheers; and a wisher:
Obey and nothing brought!

SICINIUS:
For your body that looks fault as many to my moop,
And wasting of them blood on with men's goo